# Grid tokens

The grid-world analogue of `direction_tokens.ipynb`, built with the same two-stage
pipeline (difflib over word pieces + centred, rank-normalised embeddings) over the
full Qwen3.6-35B-A3B vocabulary (248,077 tokens, 200,358 stripped forms).

Output: `grid_tokens_full_qwen3-6-35b-a3b.json`, mapping each class to raw, byte-exact vocabulary
strings — the same contract `direction_tokens_full_qwen3-6-35b-a3b.json` has, because the consumer
compares these against a jlens CSV's `top_*` columns, which come from `tok.decode([i])`.

**This notebook reads the model's vocabulary and nothing else.** No jlens CSV, no
trajectories. Deriving the vocabulary from what is popular in j-space and then measuring
how grid-loaded j-space is would be circular; `direction_tokens.ipynb` never reads a CSV
either, and that independence is what makes the measurement mean anything.

That now covers the *inputs to a score* as well as the candidate set: **every seed and
anchor must itself be a token in the model's vocabulary** (`admissible()`, cell 2), and
the lexical stage's "this word is already explained" guard reads BPE merge order instead
of `/usr/share/dict` (`ENGLISH_RANK`, cell 4). MiniLM stays, but only as the similarity
engine — it scores strings, it never supplies one. The inadmissible seeds are left in the
source and filtered at run time, so the cost of the rule is printed rather than hidden.

Classes: `WALL`, `OPEN`, `GOAL`, `AGENT`, `AXIS`, `STATUS`.

`RELATION` is deliberately absent. Over all reasoning available locally (17,434 chars of
analysis channel across the 5x5 / 11x11 / 15x15 trajectories) the words `below`, `corner`,
`edge`, `north`/`south`/`east`/`west`, `closer`, `diagonal` and `steps away` never appear;
`above` and `adjacent` appear once each. The model reasons in absolute coordinates and
cardinal directions, never egocentrically, so a relational class would be built out of
words it does not emit.

In [1]:
import numpy as np
import pandas as pd

In [2]:
# 1. the full Qwen3.6-35B-A3B vocabulary
#
# Same source as direction_tokens_qwen.ipynb: every token the model has, not a subset.
# MODEL_ID is the only thing that differs from grid_tokens.ipynb -- every seed, anchor,
# threshold and guard below is byte-identical to it, so any difference between the two
# grid vocabularies is the model and nothing else. The counts quoted in the comments
# are the gpt-oss run's record; re-read cell 8's bands before trusting SEM_T here.
#
# Two directories, because they are two different things:
#
#   CACHE_DIR  vocab_all_tokens.npy and vocab_form_embeddings.npy. Both derive from the
#              tokenizer alone -- no lens, no trajectories -- so they are shared with
#              direction_tokens_qwen.ipynb, which is why the jlens folder is on the search
#              path: that notebook already wrote them there. Both names carry MODEL_SLUG,
#              so the gpt-oss caches beside them are never picked up by mistake. The embedding matrix is
#              ~40 min of CPU, so reusing it is the whole point. Nothing is *read* from
#              the jlens folder that is not one of those two caches.
#   OUT_DIR    this notebook's own products.
#
# CACHE_DIR resolves to the first path that already exists, then falls back to something
# always writable. The old single-constant version defaulted to the workstation path and
# died in os.makedirs('/media/...') on any other machine -- which is the *default* case,
# since neither .npy is in the repo.
import os
from pathlib import Path

from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3.6-35B-A3B"

# MODEL_SLUG tags every artifact this notebook reads or writes, because none of the four
# may collide with the gpt-oss-20b ones. vocab_all_tokens.npy in particular is loaded from
# cache with no check on which tokenizer wrote it, so an unslugged run here would silently
# re-emit the gpt-oss vocabulary under a Qwen label; and the output JSON would overwrite
# the committed gpt-oss vocabulary that every published result scores against.
MODEL_SLUG = MODEL_ID.split("/")[-1].lower().replace(".", "-")  # qwen3-6-35b-a3b

_CACHE_CANDIDATES = [
    os.environ.get("INTERP_CACHE_DIR"),
    "/Users/alexdimofte/workspace/northeastern/data/jlens",  # shared with direction_tokens.ipynb
]
CACHE_DIR = Path(
    next((p for p in _CACHE_CANDIDATES if p and os.path.isdir(p)), Path.home() / ".cache" / "interp-vocab")
)


def _repo_root(start=Path.cwd()):
    """Nearest ancestor holding pyproject.toml, so OUT_DIR does not follow the kernel cwd."""
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists():
            return d
    return start


# Anchored to the repo root on purpose: .gitignore's `data/**` is root-anchored, so a
# cwd-relative "data" run from notebooks/ would land in notebooks/data/ -- not ignored,
# and committed by accident.
OUT_DIR = Path(os.environ.get("INTERP_OUT_DIR") or _repo_root() / "data")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"CACHE_DIR = {CACHE_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")

VOCAB_NPY = CACHE_DIR / f"vocab_all_tokens_{MODEL_SLUG}.npy"

if VOCAB_NPY.exists():
    list_all_tokens = np.load(VOCAB_NPY, allow_pickle=True)
    print(f"loaded cached vocabulary {VOCAB_NPY}")
else:
    _vtok = AutoTokenizer.from_pretrained(MODEL_ID)
    list_all_tokens = np.array([_vtok.decode([i]) for i in range(len(_vtok))], dtype=object)
    np.save(VOCAB_NPY, list_all_tokens)
    print(f"decoded and cached vocabulary -> {VOCAB_NPY}")

print(f"{len(list_all_tokens)} tokens, {len(set(map(str, list_all_tokens)))} unique strings")

CACHE_DIR = /Users/alexdimofte/workspace/northeastern/data/jlens
OUT_DIR   = /Users/alexdimofte/workspace/northeastern/interp/data
loaded cached vocabulary /Users/alexdimofte/workspace/northeastern/data/jlens/vocab_all_tokens_qwen3-6-35b-a3b.npy
248077 tokens, 247133 unique strings


In [3]:
# 2. token -> word pieces, and the vocabulary index everything else is scored against
#     (identical to direction_tokens.ipynb cell 2)
import re

SEP_RE = re.compile(r"""[_\-./\\:,;()\[\]{}<>"'`|!?*+=#@$%^&~\s]+""")
CAMEL_RE = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")


def strip_token(tok):
    """Surface form with whitespace/punctuation shaved off: '_LEFT' -> 'LEFT'."""
    return SEP_RE.sub(" ", tok).strip()


def parts_of(tok):
    """Word pieces to match on, lowercased: '.moveLeft' -> ['move', 'left', 'moveleft']."""
    out = []
    for piece in SEP_RE.split(tok.strip()):
        if piece:
            out.extend(p for p in CAMEL_RE.split(piece) if p)
    whole = SEP_RE.sub("", tok.strip())
    if whole:
        out.append(whole)
    return list(dict.fromkeys(p.lower() for p in out))


# VOCAB_FORMS: every token the model has, normalised the same way a seed is. A seed or
# anchor may only be a string in here. Anything else is a word we invented, and it can
# reach the vocabulary only by *inexact* match, which is where the direction run's false
# positives came from (its removed .62 tier: aristera 186, sinistra 110, descendre 99).
VOCAB_FORMS = {strip_token(str(t)).lower() for t in list_all_tokens} - {""}


def admissible(groups, kind):
    """Keep only the seeds/anchors the model has as tokens, and print what was dropped."""
    print(f"{kind}: admissible = some vocabulary token strips/lowercases to it")
    kept = {}
    for d, words in groups.items():
        kept[d] = [w for w in words if strip_token(w).lower() in VOCAB_FORMS]
        gone = [w for w in words if strip_token(w).lower() not in VOCAB_FORMS]
        print(f"  {d:7s} {len(kept[d]):2d}/{len(words):2d} kept    dropped: {gone}")
    return kept


print(f"{len(VOCAB_FORMS)} normalised forms over {len(list_all_tokens)} tokens\n")

# --- reachability of the grid symbols -------------------------------------------------
#
# The direction notebook could put the arrows straight into SEM_ANCHORS because SEP_RE
# does not contain them. That does NOT transfer to the cell symbols: '#', '_', '?', '*'
# and '+' are all inside SEP_RE, so both strip_token() and parts_of() reduce them to
# nothing and neither stage can ever score them. 'G'/'A'/'D'/'K' survive as bare letters.
#
# This is a property of the methodology, not a bug to patch here: the pipeline matches
# *words*. The symbol and coordinate signal has to be recovered structurally instead --
# a run of cell-symbol tokens as wide as the grid, or the run '(' '9' ',' '4' ')', is
# unambiguous by position even though no single token in it is.
for sym in ["#", " #", "_", " _", "G", " G", "A", "?", "*", "+", "'#", '"#']:
    print(f"  {sym!r:6s} strip={strip_token(sym)!r:6s} parts={parts_of(sym)}")

177106 normalised forms over 248077 tokens

  '#'    strip=''     parts=[]
  ' #'   strip=''     parts=[]
  '_'    strip=''     parts=[]
  ' _'   strip=''     parts=[]
  'G'    strip='G'    parts=['g']
  ' G'   strip='G'    parts=['g']
  'A'    strip='A'    parts=['a']
  '?'    strip=''     parts=[]
  '*'    strip=''     parts=[]
  '+'    strip=''     parts=[]
  "'#"   strip=''     parts=[]
  '"#'   strip=''     parts=[]


In [39]:
# 3. seeds
#
# THE RULE: every seed and every anchor must itself be a token in the model's vocabulary.
# The candidate set was always vocabulary-only; this extends that to the strings we score
# it against, so nothing outside the model decides what counts as a grid word. The lists
# below are kept whole and filtered by admissible() at the bottom rather than pruned by
# hand, so the price of the rule stays visible in the output.
#
# Two lens-independent sources, exactly as with directions:
#   - the grid legend in grid_params: # wall, _ open space, G goal, A agent,
#     D door, K key, ? unknown, * fog
#   - the reasoning surface vocabulary, from counting the analysis channel of every
#     trajectory available locally:
#       row 119 | col 96 | path 55 | open 53 | goal 34 | column 28 | G 25 |
#       agent 15 | can't 12 | blocked 10 | walls 9 | cannot 8 | obstacles 7 |
#       wall 6 | grid 6 | coordinates 5
#
# LEX_SEEDS feeds difflib and stays conservative, per the direction notebook: short and
# English-homographic forms are the ones that generate thousands of false positives.
#
# The cell-4 guards make most 4-char seeds usable: they demand an *exact* part match, so
# 'goalkeeper', 'wallpaper', 'colspan' and 'openssl' are all rejected on length alone.
# What that does NOT cover is the camel-split path -- parts_of('OpenAI') is
# ['open', 'ai', 'openai'], which contains 'open' exactly. See the note on the "open"
# class below; it is the one seed that had to be dropped.
#
# Seeds are kept disjoint across classes. 'blocked' belongs to STATUS, not WALL, so the
# lexical argmax is not deciding between two classes that seeded the same word.
LEX_SEEDS = {
    "wall": [
        "wall",
        "walls",
        "muro",
        "muri",
        "pared",
        "parede",
        "mauer",
        "wand",
        "parete",
        "стена",
        "стены",
        "duvar",
        "ściana",
        "τοίχος",
        "barrier",
        "barrera",
        "barriere",
        "obstacle",
        "obstaculo",
        "obstáculo",
        "ostacolo",
        "hindernis",
        "препятствие",
        "obstaculos",
        "obstacles",
    ],
    # 'open' is deliberately NOT a lexical seed, for the same reason 'le'/'alt'/'sol'/
    # 'port' were dropped from the direction seeds: it is a short English-homographic
    # form that fires on ordinary code tokens. Measured on the probe list in cell 9,
    # it was the only seed to produce real false positives -- parts_of() camel-splits
    # 'OpenAI' into ['open', 'ai', 'openai'], so the exact-match guard does not save it,
    # and ' OpenAI' is a token this model certainly emits (its own system prompt says
    # "trained by OpenAI"). '.open' leaked the same way. The open/empty sense is left to
    # the semantic stage, whose anchors are now single words -- see the note there.
    "open": [
        "abierto",
        "abierta",
        "ouvert",
        "ouverte",
        "offen",
        "aperto",
        "aberto",
        "открыто",
        "otwarty",
        "empty",
        "vacio",
        "vacío",
        "leer",
        "vuoto",
        "vazio",
        "пусто",
        "pusty",
        "libre",
        "frei",
        "libero",
        "livre",
        "свободно",
    ],
    "goal": [
        "goal",
        "goals",
        "objetivo",
        "objectif",
        "ziel",
        "obiettivo",
        "objectivo",
        "цель",
        "hedef",
        "στόχος",
        "target",
        "destination",
        "destino",
        "destinazione",
        "ziele",
        "G",
    ],
    "agent": ["agent", "agents", "agente", "agenten", "agenti", "агент", "ajan", "A"],
    "axis": [
        "row",
        "rows",
        "column",
        "columns",
        "col",
        "cols",
        "fila",
        "filas",
        "columna",
        "columnas",
        "ligne",
        "colonne",
        "zeile",
        "spalte",
        "riga",
        "colonna",
        "linha",
        "coluna",
        "строка",
        "столбец",
        "satır",
        "sütun",
        "wiersz",
        "kolumna",
        "coordinate",
        "coordinates",
        "coordenada",
        "coordonnee",
        "coordonnée",
        "koordinate",
        "coordinata",
        "координата",
        "position",
        "posicion",
        "posición",
        "posizione",
        "позиция",
    ],
    "status": [
        "blocked",
        "bloqueado",
        "bloccato",
        "blockiert",
        "заблокировано",
        "unreachable",
        "reachable",
        "passable",
        "impassable",
        "traversable",
        "inaccessible",
        "inaccesible",
        "unerreichbar",
    ],
}

# SEM_ANCHORS feeds the embedder, which scores meaning rather than characters. The
# direction notebook's two rules carry over.
#
# AMBIGUOUS MEANINGS. The grid vocabulary is far more homographic with code than the
# direction vocabulary was, and these are the known traps:
#   open   -> open() / opening / OpenAI / openssl
#   row    -> dataframe row / <tr> / rowspan
#   col    -> column of a table / col-md-6
#   key    -> dict key / API key / keyboard
#   goal   -> football goal / business goals
#   agent  -> user agent / HTTP agent
# The fix used to be a phrase that pins the grid sense ('open space', 'grid row', 'the
# goal cell'). Phrases are no longer available -- no phrase is a single token -- so an
# ambiguous form now either falls out of the anchor set or is watched in the cell-7 bands
# and the cell-9 probes. They are kept in the source and dropped by admissible() rather
# than deleted, so what the rule costs stays visible.
#
# 'open' is the one class the rule would empty: all eleven of its anchors were phrases,
# precisely because the bare word is the homograph, and it has no lexical seed either.
# Its anchors are therefore the admissible single words for the same sense that are NOT
# the homograph -- empty, blank, vacant. 'free' and 'void' are tokens too but are far
# more C than grid, and 'leer' / 'vide' / 'frei' / '空' were tried and dropped: they are
# homographs across languages (leer = to read in Spanish, vide = vid-, 空 = sky/air,
# frei = free-as-in-liberty) and between them admitted 82 tokens, nearly all junk.
#
# HUB ANCHORS -> the single-character symbols G/A/D/K are the extreme case of the hub
# problem cell 6 documents for '위'. They are included anyway, because rank normalisation
# is per anchor: a hub anchor cannot flood the result, only its own top percentile is
# admitted. Watch them in the band inspection regardless.
#
# NOT INCLUDED, and cell 2 explains why: '#', '_', '?', '*'. Unlike the arrows in the
# direction notebook, these characters are inside SEP_RE, so strip_token() reduces them to
# the empty string and they are unreachable by either stage.
SEM_ANCHORS = {
    "wall": [
        "wall",
        "a wall",
        "muro",
        "mur",
        "Mauer",
        "parete",
        "parede",
        "стена",
        "duvar",
        "ściana",
        "barrier",
        "obstacle",
        "brick wall",
    ],
    "open": [
        "open space",
        "empty cell",
        "free square",
        "espacio abierto",
        "case vide",
        "freies Feld",
        "spazio vuoto",
        "espaço vazio",
        "свободная клетка",
        "boş alan",
        "walkable tile",
        "empty",
        "blank",
        "vacant",
    ],
    "goal": [
        "goal",
        "the goal cell",
        "target square",
        "destination",
        "la meta",
        "objetivo",
        "objectif",
        "Ziel",
        "obiettivo",
        "цель",
        "hedef",
        "G",
    ],
    "agent": ["agent", "the agent", "current position", "agente", "Agent", "агент", "A", "player position"],
    "axis": [
        "grid row",
        "grid column",
        "row index",
        "column index",
        "coordinate",
        "fila",
        "columna",
        "ligne",
        "colonne",
        "Zeile",
        "Spalte",
        "строка",
        "столбец",
        "координата",
    ],
    "status": [
        "blocked",
        "cannot pass",
        "impassable",
        "unreachable",
        "bloqueado",
        "blockiert",
        "заблокировано",
        "no way through",
    ],
}

LEX_SEEDS = admissible(LEX_SEEDS, "LEX_SEEDS")
SEM_ANCHORS = admissible(SEM_ANCHORS, "SEM_ANCHORS")

LEX_SEEDS: admissible = some vocabulary token strips/lowercases to it
  wall    13/25 kept    dropped: ['mauer', 'duvar', 'ściana', 'τοίχος', 'barrera', 'barriere', 'obstaculo', 'obstáculo', 'ostacolo', 'hindernis', 'препятствие', 'obstaculos']
  open    17/22 kept    dropped: ['открыто', 'otwarty', 'vacio', 'vazio', 'pusty']
  goal    14/16 kept    dropped: ['objectivo', 'στόχος']
  agent    7/ 8 kept    dropped: ['agenten']
  axis    21/37 kept    dropped: ['columnas', 'zeile', 'spalte', 'riga', 'строка', 'столбец', 'satır', 'sütun', 'wiersz', 'kolumna', 'coordenada', 'coordonnee', 'coordonnée', 'koordinate', 'coordinata', 'координата']
  status   4/13 kept    dropped: ['bloqueado', 'bloccato', 'blockiert', 'заблокировано', 'passable', 'impassable', 'traversable', 'inaccesible', 'unerreichbar']
SEM_ANCHORS: admissible = some vocabulary token strips/lowercases to it
  wall     8/13 kept    dropped: ['a wall', 'Mauer', 'duvar', 'ściana', 'brick wall']
  open     3/14 kept    dropped: [

In [40]:
# 4. lexical stage: difflib over the word pieces
#
# Unchanged from direction_tokens.ipynb cell 4. One guard is left: the ratio cutoff
# scales with seed length. .80 between 5-char strings just means "one char differs", so
# short seeds demand exact equality and everything else .85. This is what makes the
# 4-char grid seeds ('open', 'goal', 'wall', 'col') usable: 'openssl'/'goalkeeper'/
# 'wallpaper'/'colspan' are rejected on length alone.
#
# The system word list that used to make an English part match at .95 is gone, here for
# the same reason as there: it was the last input to a score that did not come from the
# model. Watch the bands below and the cell-9 probes for what it used to suppress.
#
# The permissive .62 tier is deliberately NOT reinstated: it cost 726 false positives at
# 200k tokens in the direction run.
from difflib import SequenceMatcher

flat_lex = [(w, d) for d, ws in LEX_SEEDS.items() for w in ws]


def min_ratio(part, seed):
    return 1.0 if len(seed) <= 4 else 0.85  # exact only for short seeds


_part_cache = {}


def score_part(part):
    hit = _part_cache.get(part)
    if hit is not None:
        return hit
    best, bw, bd = 0.0, "", ""
    lp = len(part)
    for w, d in flat_lex:
        floor = min_ratio(part, w)
        if 2 * min(lp, len(w)) / (lp + len(w)) < floor:
            continue  # cannot reach the cutoff whatever the overlap
        r = SequenceMatcher(None, part, w).ratio()
        if r >= floor and r > best:
            best, bw, bd = r, w, d
    _part_cache[part] = (best, bw, bd)
    return _part_cache[part]


lex_score, lex_seed, lex_dir = [], [], []
for n, t in enumerate(list_all_tokens):
    best, bw, bd = 0.0, "", ""
    for part in parts_of(str(t)):
        r, w, d = score_part(part)
        if r > best:
            best, bw, bd = r, w, d
    lex_score.append(best)
    lex_seed.append(bw)
    lex_dir.append(bd)
    if n % 50000 == 0:
        print(f"  {n:>6d}/{len(list_all_tokens)}  ({len(_part_cache)} parts cached)")
lex_score = np.array(lex_score)

print(f"\n{(lex_score > 0).sum()} tokens matched lexically")
for lo, hi in [(0.99, 1.01), (0.90, 0.99), (0.85, 0.90)]:
    idx = np.where((lex_score >= lo) & (lex_score < hi))[0]
    print(f"\n=== [{lo}, {hi}) : {len(idx)} ===")
    print("   ", [repr(str(list_all_tokens[i])) for i in idx[:40]])

       0/248077  (0 parts cached)
   50000/248077  (23826 parts cached)
  100000/248077  (48251 parts cached)
  150000/248077  (97983 parts cached)
  200000/248077  (138896 parts cached)

812 tokens matched lexically

=== [0.99, 1.01) : 409 ===
    ["'row'", "' col'", "'rows'", "'col'", "' target'", "' position'", "' row'", "'Column'", "'Row'", "'empty'", "' column'", "'position'", "'Empty'", "'Position'", "' Col'", "' empty'", "'target'", "' goal'", "'Col'", "'.Column'", "'.position'", "'column'", "'Target'", "'.target'", "' rows'", "' wall'", "'pared'", "'(row'", "'agent'", "' columns'", "' agent'", "'_row'", "'(target'", "'_column'", "' goals'", "'Rows'", "' destination'", "'.empty'", "'.isEmpty'", "' Column'"]

=== [0.9, 0.99) : 111 ===
    ["'arget'", "'olumn'", "'osition'", "'ordinates'", "' positions'", "' targets'", "' Colum'", "'ouver'", "'agment'", "' accessible'", "' parsed'", "' locked'", "'ARGET'", "'OLUMN'", "' parte'", "'estination'", "'locked'", "'positions'", "'agento'

In [41]:
# 5. embedding model (CPU)
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
_tok = AutoTokenizer.from_pretrained(MODEL)
_mod = AutoModel.from_pretrained(MODEL).eval()


@torch.no_grad()
def encode(texts, batch=256, show_every=None):
    """Mean-pooled, L2-normalised embeddings. Bare strings, no template --
    'the direction "{x}"' compressed everything upward and killed the separation."""
    out = []
    for i in range(0, len(texts), batch):
        b = _tok(texts[i : i + batch], padding=True, truncation=True, max_length=16, return_tensors="pt")
        h = _mod(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).float()
        out.append(F.normalize((h * m).sum(1) / m.sum(1), dim=-1))
        if show_every and (i // batch) % show_every == 0:
            print(f"  {i + len(b['input_ids']):>6d}/{len(texts)}")
    return torch.cat(out).numpy()

In [42]:
# 6. semantic stage: centred embeddings, cosine against the anchors, rank-normalised
#
# Unchanged from direction_tokens.ipynb cell 6, including both hubness fixes:
#   TOKEN SIDE  -- subtract the mean form embedding (the space is anisotropic, so no raw
#                  cosine threshold can work: >= .85 kept 38,679 of 150,233 forms).
#   ANCHOR SIDE -- rank-normalise per anchor. z-scoring leaves each anchor a different
#                  ceiling; (cos-mean)/(1-mean) amplifies instead of damping. Rank has
#                  both properties and admits only the top percentile per anchor, which
#                  is what makes the single-letter anchors 'G'/'A' safe to include.
#
# "Which class is it?" -> raw (centred) cosine, unnormalised, as there.
EMB_NPY = CACHE_DIR / f"vocab_form_embeddings_{MODEL_SLUG}.npy"

anchor_words = [w for ws in SEM_ANCHORS.values() for w in ws]
anchor_dir = [d for d, ws in SEM_ANCHORS.items() for _ in ws]

# many tokens collapse to the same stripped form (' row', '_row', '.row')
uniq = sorted({strip_token(str(t)) for t in list_all_tokens} - {""})

# 150k forms is ~40 min on CPU, so the matrix is cached; delete the .npy to redo.
# This is the same cache direction_tokens.ipynb writes -- the forms depend only on the
# vocabulary, not on the seeds, so it is shared.
if EMB_NPY.exists():
    U = np.load(EMB_NPY)
    assert len(U) == len(uniq), f"cache is stale ({len(U)} vs {len(uniq)} forms), delete it"
    print(f"loaded cached embeddings {U.shape} from {EMB_NPY}")
else:
    U = encode(uniq, batch=256, show_every=100)
    np.save(EMB_NPY, U)

A = encode(anchor_words)


def unit(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)


centre = U.mean(0, keepdims=True)  # the common component
S = unit(U - centre) @ unit(A - centre).T  # (n_uniq, n_anchors)
RANK = S.argsort(0).argsort(0) / (len(uniq) - 1)  # percentile per anchor

i_r = RANK.argmax(axis=1)  # how strong -> threshold
i_c = S.argmax(axis=1)  # which way  -> label
by_form = {f: (float(RANK[i, i_r[i]]), anchor_words[i_c[i]], anchor_dir[i_c[i]]) for i, f in enumerate(uniq)}

sem_rank, sem_seed, sem_dir = [], [], []
for t in list_all_tokens:
    r, w, d = by_form.get(strip_token(str(t)), (0.0, "", ""))
    sem_rank.append(r)
    sem_seed.append(w)
    sem_dir.append(d)
sem_rank = np.array(sem_rank)

print("\nsanity -- the literal grid words:")
for a in ["wall", "goal", "agent", "row", "column", "open", "empty", "blocked"]:
    if a in by_form:
        s, w, d = by_form[a]
        print(f"  {a:9s} -> {d.upper():7s} (via {w!r}, rank={s:.5f})")

print("\nhub check -- how close each anchor sits to the vocabulary on average:")
for j, w in enumerate(anchor_words):
    print(f"  {w!r:22s} {anchor_dir[j]:7s} mean cos={S[:, j].mean():+.4f}")

loaded cached embeddings (200358, 384) from /Users/alexdimofte/workspace/northeastern/data/jlens/vocab_form_embeddings_qwen3-6-35b-a3b.npy


/var/folders/qs/0q06h0s57kl03gnbz11ds7cw0000gn/T/ipykernel_81014/3185030112.py:39: RuntimeWarning: divide by zero encountered in matmul
  S = unit(U - centre) @ unit(A - centre).T  # (n_uniq, n_anchors)
/var/folders/qs/0q06h0s57kl03gnbz11ds7cw0000gn/T/ipykernel_81014/3185030112.py:39: RuntimeWarning: overflow encountered in matmul
  S = unit(U - centre) @ unit(A - centre).T  # (n_uniq, n_anchors)
/var/folders/qs/0q06h0s57kl03gnbz11ds7cw0000gn/T/ipykernel_81014/3185030112.py:39: RuntimeWarning: invalid value encountered in matmul
  S = unit(U - centre) @ unit(A - centre).T  # (n_uniq, n_anchors)



sanity -- the literal grid words:
  wall      -> WALL    (via 'wall', rank=1.00000)
  goal      -> GOAL    (via 'goal', rank=1.00000)
  agent     -> AGENT   (via 'agent', rank=1.00000)
  row       -> AXIS    (via 'columna', rank=0.99989)
  column    -> AXIS    (via 'columna', rank=0.99999)
  open      -> OPEN    (via 'empty', rank=0.99195)
  empty     -> OPEN    (via 'empty', rank=1.00000)
  blocked   -> STATUS  (via 'blocked', rank=1.00000)

hub check -- how close each anchor sits to the vocabulary on average:
  'wall'                 wall    mean cos=-0.0256
  'muro'                 wall    mean cos=-0.0217
  'mur'                  wall    mean cos=-0.0102
  'parete'               wall    mean cos=+0.0072
  'parede'               wall    mean cos=-0.0261
  'стена'                wall    mean cos=-0.0155
  'barrier'              wall    mean cos=-0.0277
  'obstacle'             wall    mean cos=-0.0248
  'empty'                open    mean cos=-0.0177
  'blank'                open   

In [43]:
# 7. combine both signals into one frame
LEX_T = 0.01  # the guards in cell 4 already zeroed everything below bar
SEM_T = 0.9999  # percentile; retune from the bands printed below

df = pd.DataFrame(
    {
        "token": [str(t) for t in list_all_tokens],
        "lex_score": lex_score,
        "lex_seed": lex_seed,
        "lex_dir": lex_dir,
        "sem_rank": sem_rank,
        "sem_seed": sem_seed,
        "sem_dir": sem_dir,
    }
)
# cached so SEM_T can be retuned without recomputing anything
df.to_pickle(OUT_DIR / f"grid_scores_{MODEL_SLUG}.pkl")

print("semantic bands (where to put SEM_T):")
for lo, hi in [(0.99995, 1.01), (0.9999, 0.99995), (0.9995, 0.9999), (0.999, 0.9995), (0.998, 0.999)]:
    sel = df[(df.sem_rank >= lo) & (df.sem_rank < hi)]
    print(f"  [{lo:.5f}, {hi:.5f}) : {len(sel):5d}  {[repr(t) for t in sel.token.head(12)]}")

print(f"\nlexical  : {(df.lex_score >= LEX_T).sum():5d} tokens")
print(f"semantic : {(df.sem_rank >= SEM_T).sum():5d} tokens")
print(f"union    : {((df.lex_score >= LEX_T) | (df.sem_rank >= SEM_T)).sum():5d} tokens")

semantic bands (where to put SEM_T):
  [0.99995, 1.01000) :   455  ["'A'", "'G'", "'a'", "'g'", "' a'", "'an'", "' g'", "' A'", "' an'", "' G'", "'.A'", "'AN'"]
  [0.99990, 0.99995) :   235  ["'rows'", "'tract'", "' block'", "'location'", "'Row'", "'ã'", "' location'", "' à'", "' nothing'", "'.Location'", "'block'", "'Location'"]
  [0.99950, 0.99990) :  1975  ["'L'", "'ay'", "'un'", "' L'", "' un'", "'row'", "'ge'", "' void'", "' null'", "'gh'", "'gr'", "'ays'"]
  [0.99900, 0.99950) :  2532  ["'B'", "'J'", "'R'", "'b'", "'l'", "'r'", "' b'", "' l'", "' B'", "' R'", "' r'", "' J'"]
  [0.99800, 0.99900) :  4582  ["'F'", "'N'", "'f'", "'j'", "'m'", "'n'", "' f'", "' m'", "'ct'", "' n'", "'ly'", "' F'"]

lexical  :   812 tokens
semantic :   690 tokens
union    :  1310 tokens


In [44]:
# 8. final candidate set
review = df[(df.lex_score >= LEX_T) | (df.sem_rank >= SEM_T)].copy()
review["hit"] = np.where(
    (review.lex_score >= LEX_T) & (review.sem_rank >= SEM_T),
    "both",
    np.where(review.lex_score >= LEX_T, "lexical", "semantic"),
)
review["klass"] = np.where(review.lex_score >= LEX_T, review.lex_dir, review.sem_dir)
review["rank"] = review[["lex_score", "sem_rank"]].max(axis=1)
review = review.sort_values(["klass", "hit", "rank"], ascending=[True, True, False])

print(
    f"{len(review)} candidates  "
    f"(both={sum(review.hit == 'both')}, "
    f"lexical only={sum(review.hit == 'lexical')}, "
    f"semantic only={sum(review.hit == 'semantic')})"
)
print(pd.crosstab(review.klass, review.hit).to_string())

# Per-class precision is NOT uniform here, unlike the direction case. 'goal' and 'wall'
# as nouns should come out clean; 'axis' and 'open' sit next to dataframe rows and
# open(), so expect them to be dirtier. Reporting per class is what lets a consumer drop
# or downweight a class through the classes= argument instead of guessing.
pd.set_option("display.max_rows", 1310)
# review[["token", "klass", "hit", "lex_score", "lex_seed", "sem_rank", "sem_seed"]]

1310 candidates  (both=192, lexical only=620, semantic only=498)
hit     both  lexical  semantic
klass                          
agent     27       26        95
axis      77      265        90
goal      37      167       133
open      19       91        35
status     7       16        69
wall      25       55        76


In [45]:
# 9. contamination probes
#
# The direction notebook caught its own errors by eyeballing the bands above. These are
# the known-junk neighbours of the grid seeds; asserting they are absent turns that
# judgement call into something that fails loudly when a threshold moves. Keep reading
# the bands too -- probes catch known failures, bands catch unknown ones.
PROBES = [
    "OpenAI",
    " OpenAI",
    "openssl",
    ".open",
    "opener",
    " Opening",
    "keyboard",
    " keyword",
    "rowspan",
    "colspan",
    " DataFrame",
    " Rowling",
    "#include",
    "__init__",
    " goalkeeper",
    " wallpaper",
    " wallet",
    " Wall",
    " metadata",
]

kept = set(review.token)
leaked = [p for p in PROBES if p in kept]
print(f"{len(leaked)}/{len(PROBES)} contamination probes leaked into the candidate set")
for p in leaked:
    row = review[review.token == p].iloc[0]
    print(
        f"  {p!r:14s} -> {row.klass:7s} via {row.hit:8s} "
        f"lex={row.lex_score:.3f}({row.lex_seed!r}) sem={row.sem_rank:.5f}({row.sem_seed!r})"
    )
if not leaked:
    print("  none")

1/19 contamination probes leaked into the candidate set
  ' Wall'        -> wall    via both     lex=1.000('wall') sem=1.00000('wall')


In [51]:
for sym in ["#", " #", "_", " _", "G", " G", "A", "?", "*", "+", "'#", '"#']:
    row = review[review.token == sym].iloc[0]
    print(f"  {sym!r:6s} -> {row.klass:7s} via {row.hit:8s}")

  '#'    -> symbol  via symbol  
  ' #'   -> symbol  via symbol  
  '_'    -> symbol  via symbol  
  ' _'   -> symbol  via symbol  
  'G'    -> goal    via semantic
  ' G'   -> goal    via semantic
  'A'    -> agent   via semantic
  '?'    -> symbol  via symbol  
  '*'    -> symbol  via symbol  
  '+'    -> symbol  via symbol  
  "'#"   -> symbol  via symbol  
  '"#'   -> symbol  via symbol  


In [57]:
# 10. final output: class -> raw, unprocessed tokens
#
# Everything above (stripping, lowercasing, camel splitting) was scoring machinery only.
# What lands here is the byte-exact vocabulary string -- leading spaces, tabs, punctuation
# and all -- because the consumer compares it against a jlens CSV's top_* columns.
import json

CLASSES = ["wall", "open", "goal", "agent", "axis", "status"]

grid_tokens = {k.upper(): review.loc[review.klass == k, "token"].drop_duplicates().tolist() for k in CLASSES}

originals = set(str(t) for t in list_all_tokens)
flat = [t for v in grid_tokens.values() for t in v]
for k, v in grid_tokens.items():
    assert all(t in originals for t in v), f"{k}: token was modified"
assert len(set(flat)) == len(flat), "token assigned to two classes"

print(f"{'':8s} {'n':>5s}   sample")
for k, v in grid_tokens.items():
    print(f"{k:8s} {len(v):5d}   {[repr(t) for t in v[:8]]}")

# add the actual characters to the candidate set in the json file, because they are not reachable by either stage.
for sym, klass in [("#", "WALL"), (" #", "WALL"), ("'#", "WALL"), ('"#', "WALL"), ("_", "OPEN"), (" _", "OPEN")]:
    grid_tokens[klass].append(sym)

OUT = OUT_DIR / f"grid_tokens_full_{MODEL_SLUG}.json"
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(grid_tokens, f, ensure_ascii=False, indent=2)

# round-trip check: the file must give back byte-identical strings
with open(OUT, encoding="utf-8") as f:
    assert json.load(f) == grid_tokens
print(f"\n{len(flat)} tokens -> {OUT}")

             n   sample
WALL       156   ["' wall'", "' Wall'", "'wall'", "' barrier'", "' obstacles'", "' obstacle'", "'Wall'", "'Barrier'"]
OPEN       145   ["'empty'", "'Empty'", "' empty'", "'.empty'", "'.Empty'", "'_empty'", "' Empty'", "'(empty'"]
GOAL       337   ["' goal'", "' goals'", "' destination'", "'destination'", "'Destination'", "'goal'", "'.destination'", "' Goal'"]
AGENT      148   ["'agent'", "' agent'", "' agents'", "'Agent'", "' Agent'", "'_agent'", "'-agent'", "'.agent'"]
AXIS       432   ["'rows'", "'Column'", "'Row'", "' column'", "'.Column'", "'column'", "' rows'", "' columns'"]
STATUS      92   ["' blocked'", "'blocked'", "' Blocked'", "' unreachable'", "'_blocked'", "'Blocked'", "' blocker'", "'reachable'"]

1310 tokens -> /Users/alexdimofte/workspace/northeastern/interp/data/grid_tokens_full_qwen3-6-35b-a3b.json
